In [1]:
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os

In [4]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Set the path to your Google Drive folder
model_path = "/content/drive/MyDrive/BIA/GenAI/4 Real Life Chatbot/phi1.5"

In [3]:
def generate_answer(question, model, tokenizer):
    prompt = "Answer the following question: " + question

    # Move input to the same device as model
    device = next(model.parameters()).device
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

    with torch.no_grad():  # Disable gradient calculation for inference
        output = model.generate(
            input_ids,
            max_length=512,
            do_sample=True,
            top_k=50,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id  # Add pad token
        )[0]

    answer = tokenizer.decode(output, skip_special_tokens=True)
    end_of_text_index = answer.find("(end of text)")
    if end_of_text_index > -1:
        answer = answer[:end_of_text_index]
    return answer

def chatbot(question):
    global llm_model, tokenizer
    answer = generate_answer(question, llm_model, tokenizer)
    return answer



In [4]:
# Load the pre-trained model and tokenizer
print("Loading model from Google Drive...")
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
llm_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    trust_remote_code=True,
    local_files_only=True,
    torch_dtype=torch.float16  # Use float16 to save memory
)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
llm_model = llm_model.to(device)
print(f"Model loaded on {device}")

Loading model from Google Drive...
The repository /content/drive/MyDrive/BIA/GenAI/4 Real Life Chatbot/phi1.5 contains custom code which must be executed to correctly load the model. You can inspect the repository content at /content/drive/MyDrive/BIA/GenAI/4 Real Life Chatbot/phi1.5 .
 You can inspect the repository content at https://hf.co//content/drive/MyDrive/BIA/GenAI/4 Real Life Chatbot/phi1.5.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

Model loaded on cuda


In [7]:
# Create the Gradio interface
interface = gr.Interface(
    fn=chatbot,
    inputs=gr.Textbox(
        lines=5,  # Increased from default (usually 1-2)
        placeholder="Type your health-related question here...",
        label="Your Question"
    ),
    outputs=gr.Textbox(
        lines=10,  # Increased to show more response text
        label="AI Assistant's Answer"
    ),
    title="I am your AI Health Assistance 🏥",
    description="Ask general health-related questions to the AI Bot."
)

# Launch with share=True to get a public link
interface.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6b2010162245fbe878.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> None
Killing tunnel 127.0.0.1:7861 <> https://6b2010162245fbe878.gradio.live


In [ ]:
import gradio as gr
import os

def stop_server():
    os._exit(0)  # This will stop the Colab runtime
    return "Server stopped"

with gr.Blocks() as interface:
    gr.Markdown("# I am your AI Health Assistance 🏥")
    gr.Markdown("Ask general health-related questions to the AI Bot.")

    with gr.Row():
        question = gr.Textbox(lines=5, label="Your Question", placeholder="Type your question here...")
        answer = gr.Textbox(lines=12, label="AI Assistant's Answer")

    with gr.Row():
        submit_btn = gr.Button("Submit")
        stop_btn = gr.Button("Stop Server", variant="stop")

    submit_btn.click(fn=chatbot, inputs=question, outputs=answer)
    stop_btn.click(fn=stop_server, outputs=answer)

interface.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6609e5d2911d24d26e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
